# Bildgenerierungsanwendungen erstellen (OpenAI)

Große Sprachmodelle können mehr als nur Text generieren. Sie können auch Bilder aus Textbeschreibungen erzeugen. Bilder als Modalität sind in Medizintechnik, Architektur, Tourismus, Spieleentwicklung, Marketing und mehr nützlich. In dieser Lektion arbeiten wir mit den heutigen **GPT Image**-Modellen auf der OpenAI-Plattform.

## Lernziele

Am Ende dieser Lektion wirst du in der Lage sein:

- Zu erklären, was Bildgenerierung ist und wo sie nützlich ist.
- Die `gpt-image`-Modellfamilie zu verstehen und wie sie sich von den älteren DALL·E-Modellen unterscheidet.
- Eine Bildgenerierungsanwendung zu erstellen und Bilder zu bearbeiten.

## Was ist Bildgenerierung?

Bildgenerierungsmodelle erzeugen Bilder aus einer Texteingabe. Moderne Modelle wie `gpt-image` lernen während des Trainings die Beziehung zwischen Text und Bildern und verwandeln dann iterativ zufälliges Rauschen in ein Bild, das zu deinem Prompt passt.

Zwei bekannte Modellfamilien für Bilder sind:

- **`gpt-image` (OpenAI)** — die aktuelle Generation, die in dieser Lektion verwendet wird. Sie unterstützt Text-zu-Bild-Generierung und Bildbearbeitung (Inpainting mit Maske).
- **Midjourney** — ein beliebtes Drittanbieter-Modell mit eigenem Dienst und Workflow über Discord.

> Die älteren OpenAI-Bildmodelle — **DALL·E 2** und **DALL·E 3** — sind veraltet. DALL·E 3 ist für neue Bereitstellungen nicht mehr verfügbar, und Funktionen wie `create_variation` gab es nur in DALL·E 2. Verwende für neue Anwendungen die `gpt-image`-Modelle.

> **Wichtig:** `gpt-image`-Modelle liefern das generierte Bild als **base64** (`b64_json`), nicht als URL. Dein Code dekodiert den base64-String in Bytes und speichert ihn — es gibt keine Bild-URL zum Herunterladen.


## Erstellen Ihrer ersten Anwendung zur Bilderzeugung

Was braucht man also, um eine Anwendung zur Bilderzeugung zu erstellen? Sie benötigen die folgenden Bibliotheken:

- **python-dotenv**, es wird dringend empfohlen, diese Bibliothek zu verwenden, um Ihre Geheimnisse in einer *.env*-Datei getrennt vom Code zu speichern.
- **openai**, diese Bibliothek verwenden Sie, um mit der OpenAI-API zu interagieren.
- **pillow**, um mit Bildern in Python zu arbeiten.


1. Erstellen Sie eine Datei *.env* mit folgendem Inhalt:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Sammeln Sie die oben genannten Bibliotheken in einer Datei namens *requirements.txt* wie folgt:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Erstellen Sie als Nächstes eine virtuelle Umgebung und installieren Sie die Bibliotheken:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Für Windows verwenden Sie die folgenden Befehle, um Ihre virtuelle Umgebung zu erstellen und zu aktivieren:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Fügen Sie den folgenden Code in eine Datei namens *app.py* ein:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Lassen Sie uns diesen Code erklären:

- Zuerst importieren wir die benötigten Bibliotheken, einschließlich der OpenAI-Bibliothek, der dotenv-Bibliothek, des base64-Moduls und der Pillow-Bibliothek.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Danach erstellen wir den Client, der den API-Schlüssel aus Ihrer ``.env`` liest.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Als Nächstes generieren wir das Bild:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image`-Modelle geben das Bild als **base64**-String in `data[0].b64_json` zurück. Wir dekodieren es in Bytes und schreiben es in eine Datei — es gibt keine URL zum Herunterladen.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Schließlich öffnen wir das Bild und verwenden den Standard-Bildbetrachter, um es anzuzeigen:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Weitere Details zur Bildgenerierung

Werfen wir einen Blick auf die Parameter von `images.generate`:

- **model**, ist das zu verwendende Bildmodell (zum Beispiel `gpt-image-2`).
- **prompt**, ist der Text, der zur Generierung des Bildes verwendet wird. Hier lautet er "Hase auf Pferd, hält einen Lutscher, auf einer nebligen Wiese, auf der Narzissen wachsen".
- **size**, ist die Größe des generierten Bildes (zum Beispiel `1024x1024`, `1536x1024`, `1024x1536` oder `"auto"`).
- **n**, ist die Anzahl der generierten Bilder. Hier wird eines generiert.

> Bildmodelle verwenden keinen `temperature`-Parameter — das ist eine Steuerung für die Textgenerierung. Um Vielfalt zu erhalten, rufen Sie die API erneut auf; um Vielfalt zu reduzieren, machen Sie Ihren Prompt spezifischer.

## Zusätzliche Fähigkeiten der Bildgenerierung

Sie haben gesehen, wie man mit wenigen Zeilen Python ein Bild generiert. `gpt-image`-Modelle können auch ein bestehendes Bild **bearbeiten**. Durch Bereitstellung eines Bildes, einer optionalen **Maske** (die den zu ändernden Bereich markiert) und eines Prompts können Sie einen Teil eines Bildes verändern — zum Beispiel unserem Hasen einen Hut hinzufügen.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Das Basisbild enthält nur den Hasen; das finale Bild zeigt den Hut auf dem Hasen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
